# Preprocessing version 2
Preprocessing feels somewhat arbitrary in Scanpy compared to R. Here I will try to follow the tutorial from scirpy (TCR analysis) to merge transcriptomic and VDJ information into one object

In [ ]:
import muon as mu
import numpy as np
import pandas as pd
import scanpy as sc
import scirpy as ir
import seaborn as sb
import scrublet as scr
from matplotlib import cm as mpl_cm
from matplotlib import pyplot as plt
from muon import prot as pt
from scipy.stats import median_abs_deviation

sc.set_figure_params(figsize=(4, 4))
sc.settings.verbosity = 2  # verbosity: errors (0), warnings (1), info (2), hints (3)

## Functions

In [ ]:
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRAC', 'TRB', 'TRDV', 'TRDC', 'TRG']

def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

def filter_tcr_genes(adata):
    non_tcr_genes = adata.var_names
    for prefix in tcr_gene_prefixs:
        non_tcr_genes = [el for el in non_tcr_genes if not el.startswith(prefix)]
    adata = adata[:, non_tcr_genes]
    return adata


def filter_high_var(adata, n):
    sc.pp.highly_variable_genes(adata, n_top_genes=n)
    adata = adata[:, adata.var['highly_variable']].copy()
    return adata


def calculate_umap(adata, n_high_var=5000, remove_tcr_genes=True):
    """
    Calculates UMAP coordinates
    :param adata: AnnData object
    :param n_high_var: how many variable genes to use, None uses raw data
    :param remove_tcr_genes: whether to remove TCR genes
    :return: results stored in adata object
    """
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)            
    sc.tl.umap(adata_tmp)
    adata.obsm['X_umap'] = adata_tmp.obsm['X_umap']


def calculate_leiden(adata, resolution, n_high_var=5000, remove_tcr_genes=True):
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)
    sc.tl.leiden(adata_tmp, resolution=resolution)
    adata.obs['leiden'] = adata_tmp.obs['leiden']

# Loading data

In [ ]:
path_gex = r"..\..\02_raw_data\NK13_2\sample_filtered_feature_bc_matrix.h5"
path_tcr = r"..\..\02_raw_data\NK13_2\filtered_contig_annotations.csv"

First we load a conventional AnnData object with GEX information. We seperately load an AnnData object with TCR information and merge the two objects together into a MuData object (MultiOmicsData)

In [ ]:
## Gex
data = sc.read_10x_h5(path_gex, gex_only=False)
data.var_names_make_unique() #So that gene names are unique

## VDJ
data_tcr = ir.io.read_10x_vdj(path_tcr)


In [ ]:
data.var["feature_types"].value_counts()

### Correcting an annotation error
In the feature references, I made a copy-paste error, where sc-1OS_66 was copy pasted wrongly. I see that CellRanger generated a bunch of IDs with 66-1 to 66-22. These are probably meant to deal with dupliciated identities. I will use these, but will likely need to rerun cellranger to be certain that I didn't do any mistakes

In [ ]:
## Get a list of hashtag antibodies
TotalSeq_ann = pd.read_csv("../../03_annotation/11_multimer_experiment_ann/Feature_barcodes.csv")

## Every time there is a dublicate name in the dataframe hyphenate 1
counts = TotalSeq_ann.groupby("name").cumcount()
TotalSeq_ann["name"] = np.where(
    counts > 0,
    TotalSeq_ann["name"] + "-" + counts.astype(str),
    TotalSeq_ann["name"]
)

# Create a dictionary with ids and names
rename_dict = dict(zip(TotalSeq_ann["name"], TotalSeq_ann["id"]))

## Print the original var names
print(data.var.loc[data.var["feature_types"] == "Antibody Capture"].index.tolist())

In [ ]:
# Rename data.var
data.var_names = data.var_names.map(lambda x: rename_dict.get(x, x))

## Print the renamed var names
print(data.var.loc[data.var["feature_types"] == "Antibody Capture"].index.tolist())

### Gathering 3 types of data into a multi-omics object
Preprocessing functions came from here: https://scanpy-tutorials.readthedocs.io/en/multiomics/cite-seq/pbmc5k.html

In [ ]:
## Get a list of hashtag antibodies
TotalSeq_ann = pd.read_csv("../../03_annotation/11_multimer_experiment_ann/Feature_barcodes.csv")
HTO_list = TotalSeq_ann[TotalSeq_ann["id"].str.contains("HTO")]["id"].tolist()
Adt_list = TotalSeq_ann[~TotalSeq_ann["id"].str.contains("HTO")]["id"].tolist()

In [ ]:
protein = data[:, data.var["feature_types"] == "Antibody Capture"].copy()

# I decided to run han solo as Mimi does it, but keep the visualization aspects of dsb
# rna = data[:, data.var["feature_types"] == "Gene Expression"].copy()

#Define a list of protein features
hashtags = HTO_list
cite_seqs = Adt_list
feature_barcode_ids = hashtags + cite_seqs

# Hashtag data
# scipy has recently removed the ".A" shorthand for ".toarray()"
data.uns['hashtags'] = hashtags
for h in hashtags:
    data.obs[h] = data[:, h].X.toarray().copy()

# CiteSeq Data
data.uns['cite_ids'] = cite_seqs
for c in cite_seqs:
    data.obs[c] = data[:, c].X.toarray().copy()

# Remove Barcodes from counts
data = data[:, [gene for gene in data.var_names if gene not in feature_barcode_ids]]
data.obs['sample'] = f'NEW1'

rna = data

In [ ]:
mdata = mu.MuData({"gex": data, "airr": data_tcr, "prot": protein})

# Update 2025_7 - save the counts into a separate layer. 
mdata["gex"].layers["counts"] = mdata["gex"].X.copy()

mdata

Define a variable "prot" that acts as a shorthand for mdata["prot"]

In [ ]:
prot = mdata.mod['prot']
prot

We also need to provide the RAW droplet data for normalization for empty droplets

In [ ]:
raw_NK = r"../../02_raw_data/NK13_2/raw_feature_bc_matrix.h5"
## raw data
raw_dat = sc.read_10x_h5(raw_NK, gex_only=False)
raw_dat.var_names_make_unique() #So that gene names are unique

protein_raw = raw_dat[:, raw_dat.var["feature_types"] == "Antibody Capture"].copy()
rna_raw = raw_dat[:, raw_dat.var["feature_types"] == "Gene Expression"].copy()
mdata_raw = mu.MuData({"gex": rna_raw, "prot": protein_raw})


In [ ]:
#Calculates log10umi for each droplet in the raw data file to define where the "empty droplets" begin. 
mdata_raw['gex'].obs["log10umi"] = np.array(np.log10(mdata_raw['gex'].X.sum(axis=1) + 1)).reshape(-1)
mu.pl.histogram(mdata_raw['gex'], ['log10umi'], bins=50)
mu.pl.histogram(mdata_raw['gex'][mdata_raw['gex'].obs.log10umi >= 1], ['log10umi'], bins=50) #zoom in close

What I see here is a large peak below log10umi = 1

### Normalizing protein data
... by using information for negative droplets

Preserve original counts in a layer before the normalisation:

In [ ]:
# First save the raw count data to a new layer
prot.layers['counts'] = prot.X

See here for details on dsb / Muon workflow: https://github.com/scverse/muon-tutorials/blob/master/cite-seq/1-CITE-seq-PBMC-5k.ipynb and https://github.com/niaid/dsb

In [ ]:
mdata

In [ ]:
pt.pp.dsb(mdata, mdata_raw, empty_counts_range=(1,1), random_state=1)

In [ ]:
sc.pl.scatter(mdata['prot'], x="HTO_1", y="HTO_2")
sc.pl.scatter(mdata['prot'], x="HTO_3", y="HTO_5")
sc.pl.scatter(mdata['prot'], x="HTO_6", y="HTO_8")
sc.pl.scatter(mdata['prot'], x="sc-1OS_137", y="sc-1OS_123")  #EPLPQGQLTAY vs NLV peptide. Patient 4 has a large ELP response
sc.pl.scatter(mdata['prot'], x="sc-1OS_131", y="HTO_8")  #EPLPQGQLTAY vs NLV peptide. Patient 4 has a large ELP response
sc.pl.scatter(mdata['prot'], x="CD4_adt", y="CD8_adt")

## QC and thresholding (MAD)
Export dataframe after rudimentary QC (MT, exclusion of empty droplets and dying cells) but BEFORE exclusion of doublets

In [ ]:
#Annotate QC metrics but do not filter
mdata["gex"].var['mt'] = mdata["gex"].var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(mdata["gex"], qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
#All outliers across the three metrics
mdata["gex"].obs["outlier"] = (
    is_outlier(mdata["gex"], "total_counts", 5) |
    is_outlier(mdata["gex"], "n_genes_by_counts", 5) |
    is_outlier(mdata["gex"], "pct_counts_mt", 5)
).astype("category")
mdata["gex"].obs.outlier.value_counts()

In [ ]:
# Plot the QC metrics, and decide a cutoff %%%%%%%%%%%%
# Number of Axes & plot size
ncols = 3
nrows = 1
figsize = 4
wspace = 1

fig, axs = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(ncols * figsize + figsize * wspace * (ncols - 1), nrows * figsize),
)


# Plot individual values  
sc.pl.violin(
    mdata["gex"],
    keys="n_genes_by_counts",
    groupby='outlier',
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[0]
)
sc.pl.violin(
    mdata["gex"],
    keys="total_counts",
    groupby="outlier",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[1]
)
sc.pl.violin(
    mdata["gex"],
    keys="pct_counts_mt",
    groupby="outlier",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[2]
)

#200 lower threshold
#4000 upper threshold
#1000 upper threshold
#10% upper threshold

In [ ]:
# Plot the QC metrics, and decide a cutoff %%%%%%%%%%%%
# Number of Axes & plot size
ncols = 3
nrows = 1
figsize = 4
wspace = 1

fig, axs = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(ncols * figsize + figsize * wspace * (ncols - 1), nrows * figsize),
)

# Plot individual values  
sc.pl.violin(
    mdata["gex"], 
    keys="n_genes_by_counts",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[0]
)
sc.pl.violin(
    mdata["gex"], 
    keys="total_counts",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[1]
)
sc.pl.violin(
    mdata["gex"], 
    keys="pct_counts_mt",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[2]
)

#n_genes_total (i.e. number of genes per cell)
axs[0].axhline(y=200, linewidth = 1.5, color = "black", linestyle = "--")
axs[0].axhline(y=4000, linewidth = 1.5, color = "black", linestyle = "--")

#number of total reads 
axs[1].axhline(y=400, linewidth = 1.5, color = "black", linestyle = "--")
axs[1].axhline(y=10000, linewidth = 1.5, color = "black", linestyle = "--")

#mitochondrial gene expression
axs[2].axhline(y=10, linewidth = 1.5, color = "black", linestyle = "--")

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Update 2025_4
# Genes needs to be detected in at least 3 cells. 

# A general lower limit on the number of genes detected per cell
# CSF = 200

# A general lower limit on the total_reads detected per cell
# CSF = 400


# %%% Filter functions from muon. 
## filter_var indicates filtering on genes
mu.pp.filter_var(mdata["gex"], 'n_cells_by_counts', lambda x: x >= 3)

## filter_obs indicates filtering on observations (i.e. cells)
mu.pp.filter_obs(mdata["gex"], 'n_genes_by_counts', lambda x: (x >= 200)) 
mu.pp.filter_obs(mdata["gex"], 'n_genes_by_counts', lambda x: x < 4000)
mu.pp.filter_obs(mdata["gex"], 'total_counts', lambda x: (x > 400))
mu.pp.filter_obs(mdata["gex"], 'total_counts', lambda x: x < 10000)
mu.pp.filter_obs(mdata["gex"], 'pct_counts_mt', lambda x: x < 10)

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Normalization steps %%%%%%%%%%%%%%%%
sc.pp.normalize_total(mdata["gex"])
sc.pp.log1p(mdata["gex"])
sc.pp.highly_variable_genes(mdata["gex"], n_top_genes=5000)
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

#TCR steps %%%%%%%%%%%%%%%%%%%%%%%%%%%
ir.pp.index_chains(mdata)
ir.tl.chain_qc(mdata)    #Chain QC
mdata.obs['airr:chain_pairing'].value_counts()
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

#Update
mdata.update()

#Define NA clonotypes as "No IR detected"
mdata.obs['airr:chain_pairing'] = mdata.obs['airr:chain_pairing'].fillna('no_IR')

#Transfer the chain_pairing information
mdata["gex"].obs['chain_pairing'] = mdata["airr"].obs['chain_pairing']

With the neighbourhood graph computed, we can now perform clustering. We will use leiden clustering as an example.

In [ ]:
calculate_leiden(mdata["gex"], resolution=1, n_high_var=5000, remove_tcr_genes=True)
calculate_umap(mdata["gex"], n_high_var=5000, remove_tcr_genes=True)

In [ ]:
sc.pl.umap(mdata["gex"], color=["leiden", "total_counts", "outlier"], legend_loc="on data", size=10, legend_fontweight= 'bold', frameon= True , vmin=0, legend_fontoutline=2, cmap="RdBu_r")
sc.pl.umap(mdata["gex"], color=["chain_pairing", "CD3E", "CD8A", "CD4"], legend_loc="on data", size=40, legend_fontweight= 'bold', frameon= True,  vmin=0, legend_fontoutline=2, cmap="RdBu_r")
sc.pl.umap(mdata["gex"], color=["CX3CR1", "GZMB", "PRF1", "GZMK"], legend_loc="on data", size=40, legend_fontweight= 'bold', frameon= True, vmin=0, legend_fontoutline=2, cmap="RdBu_r")



### Protein preprocessing

In [ ]:
prot = mdata.mod['prot']
prot

# Protein preprocessing
Seperate the samples

In [ ]:
def distributions_over_columns(mdata, columns, n_rows, n_cols, y_lim=0.003, x_lim=None, title=None):
    assert len(columns) <= n_rows * n_cols, f'n_rows*n_cols={n_rows*n_cols} does not fit {len(columns)} classes.'
    fig, axes = plt.subplots(ncols=n_cols, nrows=n_rows, figsize=(n_cols * 3, n_rows * 3))
    if type(axes) == np.ndarray:
        axes = axes.reshape(-1)
    else:
        axes = [axes]

    for i, ht in enumerate(columns):
        if np.sum(np.isnan(mdata.obs[ht])) != len(mdata):
            sb.distplot(mdata[mdata.obs[ht] >= 0].obs[ht], ax=axes[i])
        axes[i].set_ylim([0, y_lim])
        if x_lim is not None and x_lim < max(mdata.obs[ht]):
            axes[i].set_xlim([0, x_lim])
        axes[i].set_title(ht)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

distributions_over_columns(mdata["gex"], hashtags, 2, 4)

In [ ]:
def hash_solo_by_sample(hashtag_cols, col_name, n_noise_barcodes):
    mdata["gex"].obs[col_name] = 'NaN'

    dfs_donor = []
    mdata["gex"].obs = mdata["gex"].obs.drop(col_name, axis=1)
    sc.external.pp.hashsolo(mdata["gex"], hashtag_cols, number_of_noise_barcodes=n_noise_barcodes)
    mdata["gex"].obs = mdata["gex"].obs.rename(columns={'Classification': col_name})

hash_solo_by_sample(hashtags, 'pool', n_noise_barcodes=7)  #n_noise_barcodes must be at least 1 less than the total number of barcodes present. 
mdata["gex"].obs['pool'].value_counts()

In [ ]:
for h in hashtags:
    mdata["gex"].obs[f'log_{h}'] = np.log(mdata["gex"].obs[h].values+1)
sb.violinplot(data=mdata["gex"].obs[[f'log_{h}' for h in hashtags]], density_norm='area')

### Double check the performance of hash_solo

In [ ]:
mdata.update()

In [ ]:
sc.pl.umap(mdata["gex"], color=['pool'])
sc.pl.umap(mdata["gex"], color=['SELL', "CCR7", "GZMK", "GZMB", "ISG15", "IRF7"])

In [ ]:
mdata["prot"].obs["pool"] = mdata["gex"].obs["pool"].copy() 
sc.pl.scatter(mdata['prot'], x="HTO_1", y="HTO_2", color="pool", size=30)

## Remove hashtag doublets and negatives

In [ ]:
#Remove observable doublets
print('Before Filtering Cluster: ', len(rna))
element_remove = ['Negative', "Doublet"]
mu.pp.filter_obs(rna, 'pool', lambda x: (x != "Doublet") & (x != "Negative"))
print('After Filtering Cluster: ', len(rna))

Inspect observable and unobservable duoblets with double TCR information

In [ ]:
mdata["gex"].obs['chain_pairing'] = mdata["airr"].obs['chain_pairing']
print(mdata["airr"].obs['chain_pairing'].value_counts())
print(mdata["gex"].obs['chain_pairing'].value_counts())
mdata.update()

two "full chains" dropped markedly by removing observable duoblets. Let's remove the remaining "two-full chain" cells

In [ ]:
#Remove assumed unobservable doublets
print("Before filtering: ", mdata.obs['airr:chain_pairing'].value_counts())
mu.pp.filter_obs(mdata, "airr:chain_pairing", lambda x: x != "two full chains") #Let's keep these in
print("After filtering: ", mdata.obs['airr:chain_pairing'].value_counts())
mdata.update()

In [ ]:
sc.pl.umap(mdata["gex"], color='pool')

### CITEseq visual check

In [ ]:
mu.pl.embedding(mdata, basis="gex:umap", color=["prot:CD4_adt", "prot:CD8_adt", "prot:CD45RA_adt", "prot:CD62L_adt", "prot:CCR7_adt","prot:CD95_adt", 
                                                "prot:CXCR3_adt", "prot:CXCR4_adt", "prot:GPR183_adt", "prot:CD27_adt", "gex:GZMK"], legend_loc="on data", legend_fontweight= 'bold', frameon= True, size=40, vmin=0, legend_fontoutline=2, cmap="RdBu_r")

                                                

## Annotate donors

In [ ]:
# Dictionary from Philipp Bruno's email 
hto_annotation = {
    "HTO_1": {"donor": "Pt20"},
    "HTO_2": {"donor": "Pt21"},
    "HTO_3": {"donor": "Pt14"},
    # "HTO_4": {"donor": "SPF"},
    "HTO_5": {"donor": "Pt18"},
    "HTO_6": {"donor": "Pt19"},
    # "HTO_7": {"donor": "Wildling"},
    "HTO_8": {"donor": "Pt17/Pt21"},  #A0201 responses will be from 21, non-A0201 responses will be from Pt17
}

# Temporarily move the index into a column
mdata["gex"].obs["tmp_index"] = mdata["gex"].obs.index

# Merge with HTO annotation
hto_df = (
    pd.DataFrame.from_dict(hto_annotation, orient="index")
      .rename_axis("HTO")
      .reset_index()
)

mdata["gex"].obs = (
    mdata["gex"].obs
    .merge(
        hto_df,
        how="left",
        left_on="pool",   # adjust if your obs column has a different name
        right_on="HTO"
    )
    .set_index("tmp_index")   # restore original index
)

# Restore index name
mdata["gex"].obs.index.name = None  # or keep the original name if needed

# Optional: make donor categorical
mdata["gex"].obs["donor"] = mdata["gex"].obs["donor"].astype("category")


In [ ]:
sc.pl.umap(mdata["gex"], color='donor')

## Save your progress

In [ ]:
mdata.write(filename=r"C:\Users\nipag\coding\MS.schober\04_data_objects\01_preprocessed_scseq_scanpy\2025_12 - with EBV\NK13_2.h5mu")